# Centro de estética — Consultas SQL

Práctica 5, Ejercicio 1 de Introducción a Base de Datos (UNLP). Las tablas se crean en una base DuckDB en memoria con datos de ejemplo inventados, y cada consulta se ejecuta mostrando su resultado.

Las consultas se ejecutan en orden: la 5 es un `UPDATE`, así que las siguientes ven el precio de la tintura ya aumentado.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from consultas import crear_base, cargar_consultas, ejecutar

con = crear_base("centro_estetica")
consultas = cargar_consultas("centro_estetica")

In [2]:
# Cantidad de filas de cada tabla
con.sql("SELECT table_name AS tabla, estimated_size AS filas FROM duckdb_tables() ORDER BY table_name").df()

,tabla,filas
0,Aplicacion,9
1,Cliente,5
2,Esteticista,4
3,Producto,6
4,ProductoAplicado,15


## Consulta 1

**Qué cambió respecto de mi resolución en papel:** había escrito `YEAR(a.fecha = 2019)`, con el paréntesis mal cerrado; lo correcto es `YEAR(a.fecha) = 2019`. Además, la subconsulta usaba `a.fecha` (el alias de la consulta de afuera) en lugar de `a2.fecha`.

In [3]:
ejecutar(con, consultas, 1)

1. Esteticistas que atendieron solamente durante 2019, ordenados por nombre, apellido y DNI

SELECT DISTINCT e.nombre, e.apellido, e.DNI, e.fecha_nac, e.especialidad
FROM Esteticista e
INNER JOIN Aplicacion a ON (e.codEst = a.codEst)
WHERE (YEAR(a.fecha) = 2019) AND e.codEst NOT IN (
    SELECT a2.codEst
    FROM Aplicacion a2
    WHERE (YEAR(a2.fecha) <> 2019)
)
ORDER BY e.nombre ASC, e.apellido ASC, e.DNI ASC; 



,nombre,apellido,DNI,fecha_nac,especialidad
0,Ana,Gómez,28111000,1985-04-12,Coloración


## Consulta 2

**Qué cambió respecto de mi resolución en papel:** mi versión hacía el promedio después de los `JOIN`, así que un cliente aparecía una vez por cada producto aplicado y su edad contaba varias veces. Con `IN` cada cliente cuenta una sola vez (ver la comparación al final).

In [4]:
ejecutar(con, consultas, 2)

2. Edad promedio de los clientes a los que se les aplicó algún producto cuyo nombre termina en 'ura'

SELECT AVG(c.edad) AS promedio_edad
FROM Cliente c
WHERE c.codCliente IN (
    SELECT a.codCliente
    FROM Aplicacion a
    INNER JOIN ProductoAplicado pa ON (a.nroAplicacion = pa.nroAplicacion)
    INNER JOIN Producto p ON (pa.codProd = p.codProd)
    WHERE (p.nombreP LIKE '%ura')
); 



,promedio_edad
0,30.0


## Consulta 3

**Qué cambió respecto de mi resolución en papel:** la lógica estaba bien: `LEFT JOIN` con `COUNT(columna)` cuenta 0 para los productos que nunca se aplicaron. Solo unifiqué los nombres de columnas sin tilde (`descripcion`), como en el esquema.

In [5]:
ejecutar(con, consultas, 3)

3. Cantidad de veces que se aplicó cada producto (incluidos los que nunca se aplicaron), de menor a mayor

SELECT p.nombreP, p.descripcion, p.stock, p.precio, COUNT(pa.nroAplicacion) AS cantidad
FROM Producto p
LEFT JOIN ProductoAplicado pa ON (p.codProd = pa.codProd)
GROUP BY p.codProd, p.nombreP, p.descripcion, p.stock, p.precio
ORDER BY cantidad ASC, p.nombreP ASC; 



,nombreP,descripcion,stock,precio,cantidad
0,mascarilla,Mascarilla facial,15,800.0,0
1,crema hidratante,Crema hidratante de manos,30,700.0,2
2,pedicura,Kit de pedicura,20,1100.0,2
3,manicura,Kit de manicura,25,900.0,3
4,esmalte,Esmalte semipermanente,60,450.0,4
5,tintura,Tintura permanente,40,1200.0,4


## Consulta 4

**Qué cambió respecto de mi resolución en papel:** estaba bien. Agregué un `ORDER BY` para que el resultado salga siempre en el mismo orden.

In [6]:
ejecutar(con, consultas, 4)

4. Esteticistas que nunca atendieron a clientes menores de 25 años

SELECT e.nombre, e.apellido, e.DNI, e.fecha_nac, e.especialidad
FROM Esteticista e
WHERE e.codEst NOT IN (
    SELECT a.codEst
    FROM Aplicacion a
    INNER JOIN Cliente c ON (c.codCliente = a.codCliente)
    WHERE (c.edad < 25)
)
ORDER BY e.apellido, e.nombre; 



,nombre,apellido,DNI,fecha_nac,especialidad
0,Bruno,Díaz,31222000,1990-09-30,Manicuría
1,Diego,Paz,36444000,1997-06-05,Cosmetología
2,Carla,Ruiz,33333000,1993-01-18,Pedicuría


## Consulta 5

**Qué cambió respecto de mi resolución en papel:** `SET (p.precio * 1.20)` no indicaba qué columna se modifica; lo correcto es `SET precio = precio * 1.20`. Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`).

In [7]:
ejecutar(con, consultas, 5)

5. Aumentar un 20 % el precio del producto 'tintura'

UPDATE Producto
SET precio = precio * 1.20
WHERE (nombreP = 'tintura'); 

Filas afectadas: 1


## Consulta 6

**Qué cambió respecto de mi resolución en papel:** tenía dos `WHERE` seguidos (el segundo tiene que ser `AND`) y faltaba cerrar el paréntesis de la subconsulta.

In [8]:
ejecutar(con, consultas, 6)

6. Clientes que fueron atendidos en 2018 y no en 2019

SELECT DISTINCT c.nombreYAp, c.DNI, c.telefono, c.direccion, c.sexo, c.edad
FROM Cliente c
INNER JOIN Aplicacion a ON (c.codCliente = a.codCliente)
WHERE (YEAR(a.fecha) = 2018) AND c.codCliente NOT IN (
    SELECT a2.codCliente
    FROM Aplicacion a2
    WHERE (YEAR(a2.fecha) = 2019)
); 



,nombreYAp,DNI,telefono,direccion,sexo,edad
0,Martín López,30222333,221-555-0102,"Calle 50 N° 456, La Plata",Masculino,34
1,Julián Castro,35444555,221-555-0104,"Calle 12 N° 321, City Bell",Masculino,29


## Consulta 7

**Qué cambió respecto de mi resolución en papel:** Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`). Saqué los `DISTINCT` porque `INTERSECT` ya elimina los repetidos.

In [9]:
ejecutar(con, consultas, 7)

7. Productos aplicados a clientas (sexo femenino) que además alguna vez se aplicaron a un precio menor a $500

SELECT p.nombreP, p.descripcion, p.stock, p.precio
FROM Producto p
INNER JOIN ProductoAplicado pa ON (p.codProd = pa.codProd)
INNER JOIN Aplicacion a ON (pa.nroAplicacion = a.nroAplicacion)
INNER JOIN Cliente c ON (a.codCliente = c.codCliente)
WHERE (c.sexo = 'Femenino')
INTERSECT
SELECT p.nombreP, p.descripcion, p.stock, p.precio
FROM Producto p
INNER JOIN ProductoAplicado pa ON (p.codProd = pa.codProd)
WHERE (pa.precio < 500); 



,nombreP,descripcion,stock,precio
0,esmalte,Esmalte semipermanente,60,450.0


## Consulta 8

**Qué cambió respecto de mi resolución en papel:** estaba bien.

In [10]:
ejecutar(con, consultas, 8)

8. Productos aplicados al cliente con DNI 38329663

SELECT DISTINCT p.nombreP, p.descripcion, p.stock, p.precio
FROM Producto p
INNER JOIN ProductoAplicado pa ON (p.codProd = pa.codProd)
INNER JOIN Aplicacion a ON (pa.nroAplicacion = a.nroAplicacion)
INNER JOIN Cliente c ON (a.codCliente = c.codCliente)
WHERE (c.DNI = 38329663); 



,nombreP,descripcion,stock,precio
0,pedicura,Kit de pedicura,20,1100.0
1,esmalte,Esmalte semipermanente,60,450.0
2,crema hidratante,Crema hidratante de manos,30,700.0
3,manicura,Kit de manicura,25,900.0


## Consulta 9

**Qué cambió respecto de mi resolución en papel:** a `ProductoAplicado` le había puesto el alias `p`, el mismo que `Producto`, y después usaba `pa`. Con el alias `pa` la consulta queda bien.

In [11]:
ejecutar(con, consultas, 9)

9. Productos que fueron aplicados por todas las esteticistas

SELECT p.nombreP, p.descripcion, p.stock, p.precio
FROM Producto p
WHERE NOT EXISTS (
    SELECT *
    FROM Esteticista e
    WHERE NOT EXISTS (
        SELECT *
        FROM Aplicacion a
        INNER JOIN ProductoAplicado pa ON (a.nroAplicacion = pa.nroAplicacion)
        WHERE (a.codEst = e.codEst) AND (pa.codProd = p.codProd)
    )
); 



,nombreP,descripcion,stock,precio
0,esmalte,Esmalte semipermanente,60,450.0


## Consulta 10

**Qué cambió respecto de mi resolución en papel:** consultaba una tabla `Venta` que no existe en este esquema (es `Aplicacion`). Además usaba `> ALL`: como la subconsulta incluye al propio cliente, nadie puede ser mayor que sí mismo y el resultado quedaba vacío. Con `>= ALL` aparece el que más gastó.

In [12]:
ejecutar(con, consultas, 10)

10. Cliente que más dinero gastó en el centro de estética

SELECT c.nombreYAp, c.DNI, c.telefono, c.direccion, c.sexo, c.edad, SUM(a.costoTotal) AS total_gastado
FROM Cliente c
INNER JOIN Aplicacion a ON (c.codCliente = a.codCliente)
GROUP BY c.codCliente, c.nombreYAp, c.DNI, c.telefono, c.direccion, c.sexo, c.edad
HAVING SUM(a.costoTotal) >= ALL (
    SELECT SUM(a2.costoTotal)
    FROM Aplicacion a2
    GROUP BY a2.codCliente
); 



,nombreYAp,DNI,telefono,direccion,sexo,edad,total_gastado
0,Laura Fernández,40111222,221-555-0101,"Calle 7 N° 1234, La Plata",Femenino,23,4200.0


## Por qué importaban las correcciones

Estas celdas ejecutan algunas versiones originales sobre una base nueva, para ver qué pasaba sin la corrección.

**Consulta 2 sin la corrección:** el promedio sale distinto porque las edades de los clientes con varios productos se cuentan más de una vez.

In [13]:
base = crear_base("centro_estetica")
base.sql("""
SELECT AVG(c.edad) AS promedio_edad
FROM Cliente c
INNER JOIN Aplicacion a ON (c.codCliente = a.codCliente)
INNER JOIN ProductoAplicado pa ON (a.nroAplicacion = pa.nroAplicacion)
INNER JOIN Producto p ON (pa.codProd = p.codProd)
WHERE (p.nombreP LIKE '%ura')
""").df()

,promedio_edad
0,29.222222


**Consulta 10 con `> ALL`:** no devuelve ningún cliente.

In [14]:
base.sql("""
SELECT c.nombreYAp, SUM(a.costoTotal) AS total_gastado
FROM Cliente c
INNER JOIN Aplicacion a ON (c.codCliente = a.codCliente)
GROUP BY c.codCliente, c.nombreYAp
HAVING SUM(a.costoTotal) > ALL (
    SELECT SUM(a2.costoTotal)
    FROM Aplicacion a2
    GROUP BY a2.codCliente
)
""").df()

,nombreYAp,total_gastado
